# Easy-Irodori-TTS v1.2 (Google Colab版)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/izum00/Easy-Irodori-TTS_v1.2-colab/blob/main/Easy_Irodori_TTS_v1.2_Colab.ipynb)

Irodori-TTS を Colab の GPU で使うためのノートブックです。**ベース / Anime / MF（高速）** の全モデルに対応しています。

**使い方**
1. メニュー「ランタイム → ランタイムのタイプを変更」で **GPU（T4 など）** を選ぶ
2. 下のセルを **上から順に実行**（①セットアップ → ② 任意：Googleドライブ → ③ 起動）
3. ③の出力に表示される `https://xxxx.gradio.live` を開く

初回のセットアップには数分〜十数分かかります（PyTorch等のダウンロード）。モデルは、画面で初めて選んだときに取得されます。

> 制作：ゆうぷろ with Codex ／ 音声エンジン・ベース・MF：[Aratako / Irodori-TTS](https://github.com/Aratako/Irodori-TTS) ／ Anime：[phasefield-audio](https://huggingface.co/phasefield-audio/Irodori-TTS-v4.1-Anime)
>
> 本人の同意なしの声の複製・公開、なりすまし、人をだます目的での利用は禁止です。生成物は利用者の責任で使用してください。

## ① セットアップ（最初に1回）

In [ ]:
#@title ① セットアップ（GitHubからダウンロード → 実行環境の準備）
import os, subprocess, sys, shutil
from pathlib import Path

REPO_URL   = "https://github.com/izum00/Easy-Irodori-TTS_v1.2-colab.git"
APP_DIR    = Path("/content/Easy-Irodori-TTS_v1.2-colab")
UPSTREAM   = "https://github.com/Aratako/Irodori-TTS.git"
REVISION   = "89f9d8fbd4d51ea019867ee1197725ede1df13c5"   # 動作確認済みの固定リビジョン
VENV_DIR   = Path("/content/irodori_venv")

def sh(cmd, cwd=None, env=None):
    """コマンドを実行し、出力をそのまま表示する。失敗したら例外。"""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), cwd=cwd, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"コマンドが失敗しました ({p.returncode}): {cmd}")

# --- GPU確認 -------------------------------------------------------------
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("GPUが見つかりません。「ランタイム → ランタイムのタイプを変更」でGPUを選び、"
                       "再接続してからこのセルを実行してください。")
print("GPU:", gpu.stdout.strip())

# --- 1) 本体をGitHubからダウンロード --------------------------------------
if (APP_DIR / ".git").exists():
    sh(["git", "-C", str(APP_DIR), "pull", "--ff-only"])
else:
    if APP_DIR.exists():
        shutil.rmtree(APP_DIR)
    sh(["git", "clone", "--depth", "1", REPO_URL, str(APP_DIR)])

# --- 2) ffmpeg（話速変更・音声デコードに使用）と uv ------------------------
if not shutil.which("ffmpeg"):
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg")
sh([sys.executable, "-m", "pip", "install", "-q", "uv"])

# --- 3) 本家 Irodori-TTS を固定リビジョンで取得 ----------------------------
src = APP_DIR / "Irodori-TTS"
if not (src / ".git").exists():
    src.mkdir(exist_ok=True)
    sh(["git", "init", str(src)])
marker = src / ".easy-source-ready"
if not marker.exists() or marker.read_text().strip() != REVISION:
    sh(["git", "-C", str(src), "fetch", "--depth", "1", UPSTREAM, REVISION])
    sh(["git", "-C", str(src), "checkout", "--detach", REVISION])
    marker.write_text(REVISION)

# --- 4) 独立したPython 3.11環境にロックファイル通りに依存関係を導入 ----------
env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(VENV_DIR), UV_LINK_MODE="copy")
sh([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--extra", "cu128", "--python", "3.11"],
   cwd=src, env=env)

# --- 5) 動作チェック ----------------------------------------------------
check_env = dict(os.environ, EASY_IRODORI_ROOT=str(APP_DIR), PYTHONUTF8="1")
sh([str(VENV_DIR / "bin" / "python"), str(APP_DIR / "easy_launcher.py"), "--check"], env=check_env)
print("\n✅ セットアップ完了。次のセルへ進んでください。")

## ② Googleドライブに音声を自動保存（チェックで切り替え）
初期値は **ON（ドライブに接続）** です。接続したくない場合は、セルのチェックボックス `USE_DRIVE` を外して実行してください（スキップしても構いません）。
OFFの場合、生成した音声はセッション終了で消えるため、必要なものは画面の「WAVをダウンロード」で保存してください。

In [ ]:
#@title ② Googleドライブに保存する（チェックを外すと接続しません）
#@markdown チェックON：ドライブに接続し、音声を自動保存します（初期値）
USE_DRIVE = True  #@param {type:"boolean"}
#@markdown 保存先フォルダ（マイドライブ内）
DRIVE_FOLDER = "IrodoriTTS"  #@param {type:"string"}

import json
from pathlib import Path

cfg = Path("/content/Easy-Irodori-TTS_v1.2-colab/config/settings.json")
cfg.parent.mkdir(parents=True, exist_ok=True)
data = json.loads(cfg.read_text(encoding="utf-8")) if cfg.exists() else {}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    out_dir.mkdir(parents=True, exist_ok=True)
    data.update(custom_output=True, output_dir=str(out_dir), auto_export=True)
    print("✅ 保存先:", out_dir)
else:
    # 前回のドライブ設定が残らないよう、Colab内の標準の保存先へ戻す
    data.update(custom_output=False, output_dir="")
    print("ドライブには接続しません。音声はColab内（outputs/exports）に保存されます。"
          "セッション終了で消えるので、必要な音声はブラウザの「WAVをダウンロード」で保存してください。")
cfg.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

## ③ 起動
出力に表示される **`Running on public URL: https://〜.gradio.live`** をクリックして開きます（`share=True`）。
止めるときはセルの停止ボタン（■）を押してください。

In [ ]:
#@title ③ 起動（公開リンクが表示されます）
import os, subprocess
from pathlib import Path

APP_DIR  = Path("/content/Easy-Irodori-TTS_v1.2-colab")
VENV_PY  = "/content/irodori_venv/bin/python"

env = dict(os.environ,
           EASY_IRODORI_ROOT=str(APP_DIR),
           PYTHONUTF8="1", PYTHONUNBUFFERED="1",
           GRADIO_ANALYTICS_ENABLED="False", HF_HUB_DISABLE_TELEMETRY="1", WANDB_MODE="disabled",
           HF_HOME="/content/hf_home")

proc = subprocess.Popen([VENV_PY, str(APP_DIR / "easy_launcher.py"), "--port", "7860"],
                        cwd=str(APP_DIR), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\n停止しています...")
finally:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()

## 困ったとき
- **GPUが見つからない**：「ランタイム → ランタイムのタイプを変更」でGPUを選び、再接続してください。
- **GPUメモリ不足**：画面の「モデルをメモリから解放」を押す／候補数や文章を減らしてください。
- **公開リンクが開かない**：③のセルを停止して再実行すると、新しいリンクが発行されます（リンクは約1週間で失効します）。
- **更新を取り込みたい**：①のセルをもう一度実行すると、GitHubの最新版に更新されます。
- **MFモデル**：標準は4ステップです。増やしすぎないでください。